## Setup & Initialization

In [1]:
import sys

# Remove the old module if it was already imported
if 'ConnectionConfig' in sys.modules:
    del sys.modules['ConnectionConfig']

# Now import it freshly
import importlib
import ConnectionConfig as cc
importlib.reload(cc)

# Verify which file is loaded
print(cc.__file__)

/home/jovyan/work/02-treasure-hunt-dwh/ConnectionConfig.py


In [2]:
import random
import pyspark
from delta.tables import DeltaTable, IdentityGenerator
from pyspark.sql import SparkSession, functions as F
import ConnectionConfig as cc
from importlib import reload

reload(cc)
from pyspark.sql.functions import *

print(cc.__file__)

cc.setupEnvironment()
cc.listEnvironment()

/home/jovyan/work/02-treasure-hunt-dwh/ConnectionConfig.py
Running in Docker environment.
Setting JAVA_HOME to /usr/lib/jvm/java-17-openjdk-arm64
SHELL: /bin/bash
CONDA_EXE: /opt/conda/bin/conda
_CE_M: 
HOSTNAME: sparkjupyter-data4
LANGUAGE: C.UTF-8
JAVA_HOME: /usr/lib/jvm/java-17-openjdk-arm64
_START_SH_EXECUTED: 1
SPARK_OPTS: --driver-java-options=-Xms1024M --driver-java-options=-Xmx4096M --driver-java-options=-Dlog4j.logLevel=info
NB_UID: 1000
XML_CATALOG_FILES: file:///opt/conda/etc/xml/catalog file:///etc/xml/catalog
PWD: /home/jovyan
CONDA_PREFIX: /opt/conda
PYSPARK_SUBMIT_ARGS: pyspark-shell
HOME: /home/jovyan
LANG: C.UTF-8
NB_GID: 100
CONDA_PROMPT_MODIFIER: (base) 
_CONDA_EXE: /opt/conda/bin/conda
_CONDA_ROOT: /opt/conda
PYTHONPATH: /usr/local/spark/python/lib/py4j-0.10.9.9-src.zip:/usr/local/spark/python:
_CE_CONDA: 
CONDA_SHLVL: 1
SHLVL: 0
CONDA_DIR: /opt/conda
SPARK_HOME: /usr/local/spark
CONDA_PYTHON_EXE: /opt/conda/bin/python
JUPYTER_PORT: 8888
SPARK_CONF_DIR: /usr/local/s

In [3]:
spark = cc.startLocalCluster("factTreasureHunt")
spark.getActiveSession()

Running in Docker: using pre-installed Spark + JARs


## Reading the source table

In [4]:
cc.set_connectionProfile("catchem")

print(cc.create_jdbc())

jdbc:postgresql://postgres-data4:5432/catchem?user=postgres&password=data&ssl=false


In [5]:
#EXTRACT

# Load DataFrames
df_operational_user = cc.read_postgres_without_partition(spark, "user_table")
df_operational_treasure = cc.read_postgres_without_partition(spark, "treasure")
df_operational_treasure_log = cc.read_postgres_by_date(spark, "treasure_log", "log_time")
df_operational_city = cc.read_postgres_without_partition(spark, "city")
df_operational_country = cc.read_postgres_without_partition(spark, "country")
df_operational_treasure_stages = cc.read_postgres_without_partition(spark, "treasure_stages")

print('user size: ', df_operational_user.count())  ## 486523
print('treasure size: ', df_operational_treasure.count())  ## 128294
print('treasure_log size: ', df_operational_treasure_log.count())  ## 1789438
print('city size: ', df_operational_city.count())  ## 1264588
print('country size: ', df_operational_country.count())  ## 249
print('treasure_stages size: ', df_operational_treasure_stages.count()) ## 504119


user size:  486523
treasure size:  128294
treasure_log size:  1789438
city size:  1264588
country size:  249
treasure_stages size:  504119


## Transforming the source to the dimension format

In [6]:
#TRANSFORM
df_operational_city = df_operational_city.select(
    "city_id",
    "city_name",
    "postal_code",
    F.col("country_code").alias("country_id")
)


df_operational_country = df_operational_country.select(
    F.col("code").alias("country_id"),
    F.col("name").alias("country_name"),

)

df_operational_user = df_operational_user.select(
    F.col("id").alias("user_id"),
    F.col("city_city_id").alias("city_id")
)



df_operational_treasure_log = (
    df_operational_treasure_log
    .filter(col("log_time").isNotNull())
    .select(
        col("hunter_id").alias("user_id"),
        ((unix_timestamp("log_time") - unix_timestamp("session_start")) / 60)
            .alias("duration_minutes"),
        col("treasure_id"),
        date_trunc("hour", col("log_time")).alias("log_hour"),
        F.to_date("log_time").alias("full_date")
    )
)



# 2. Transform Treasure
# Logic: Simple column selection and aliasing
df_operational_treasure = df_operational_treasure.select(
    F.col("id").alias("treasure_id"),
    "difficulty",
    "terrain"
)

# 3. Transform Treasure Stages
# Logic: Group by treasure and count the stages
df_operational_treasure_stages = df_operational_treasure_stages \
    .groupBy("treasure_id") \
    .agg(F.count("stages_id").alias("stage_count"))

df_operational_treasure = (
    df_operational_treasure
    .join(df_operational_treasure_stages, "treasure_id", "left")
    .select(
        "treasure_id",
        "difficulty",
        "terrain",
        F.coalesce(F.col("stage_count"), F.lit(0)).cast("integer").alias("stage_count")
    )
)

# User + location (postal code)
df_user_with_location = df_operational_user.join(
    df_operational_city.select("city_id", "postal_code"),
    "city_id",
    "left"
)





df_operational_treasure.show(5, truncate=False)

+-------------------------------------------------+----------+-------+-----------+
|treasure_id                                      |difficulty|terrain|stage_count|
+-------------------------------------------------+----------+-------+-----------+
|[00 03 72 3C 7C C3 42 62 A6 F3 E8 A3 E7 B8 05 63]|3         |2      |7          |
|[00 04 62 1B 29 E8 48 DB 8D 98 C6 95 47 E5 1D A0]|1         |2      |4          |
|[00 05 1E A2 05 87 47 1F 80 A3 28 81 99 C8 21 D3]|0         |2      |1          |
|[00 00 3E 2C B1 40 41 B1 97 F0 00 33 C0 97 E0 70]|3         |1      |4          |
|[00 04 39 A0 98 75 40 4F 9F EE FA 1A 40 E9 C7 54]|4         |4      |1          |
+-------------------------------------------------+----------+-------+-----------+
only showing top 5 rows


In [ ]:
#df_operational_treasure_log2.show(5)

## Creating the fact table

In [ ]:
from pyspark.sql.types import LongType, IntegerType, TimestampType

# Initialize the Fact table
DeltaTable.createOrReplace(spark) \
    .tableName("factTreasureHunt") \
    .addColumn("treasure_hunt_sk", LongType(), nullable=False, generatedAlwaysAs=IdentityGenerator(0, 1)) \
    .addColumn("user_sk", LongType(), nullable=False) \
    .addColumn("treasure_type_sk", IntegerType(), nullable=False) \
    .addColumn("date_sk", LongType(), nullable=False) \
    .addColumn("season_sk", IntegerType(), nullable=False) \
    .addColumn("hour", IntegerType(), nullable=False) \
    .addColumn("weather_sk", LongType(), nullable=False) \
    .addColumn("duration_minutes", IntegerType(), nullable=False) \
    .property("delta.feature.identityColumns", "supported") \
    .execute()

## Deleting the dimension table if it already exists

In [ ]:
cc.delete_delta_table(spark, "factTreasureHunt")

## Populating the dimension table


In [ ]:
## Load dimension tables to get surrogate keys
dim_user = spark.read.format("delta").load("spark-warehouse/dim_user")
dim_treasure = spark.read.format("delta").load("spark-warehouse/dim_treasure")
dim_date = spark.read.format("delta").load("spark-warehouse/dimdate")
dim_season = spark.read.format("delta").load("spark-warehouse/dimseason")
dim_weather = spark.read.format("delta").load("spark-warehouse/dim_weather2")


Extracting the json files

In [7]:
# Load weather JSON files
weather_json_path = "./data/weather_data/*.json"
weather_json = spark.read.option("multiLine", True).json(weather_json_path)

In [9]:
weather_exploded = weather_json.withColumn("weather_entry", explode(col("weather")))

# Convert dt (Unix epoch) to timestamp
weather_df = weather_exploded.select(
    col("zipCode").alias("postal_code"),
    to_timestamp(from_unixtime(col("dt"))).alias("datetime"),
    col("weather_entry.id").alias("weather_id")
)

# Round timestamp down to the nearest hour
weather_df = weather_df.withColumn(
    "datetime_hour",
    date_trunc("hour", col("datetime"))
)


weather_df.show(10, truncate=False)

+-----------+-------------------+----------+-------------------+
|postal_code|datetime           |weather_id|datetime_hour      |
+-----------+-------------------+----------+-------------------+
|9176       |2020-10-10 10:39:15|200       |2020-10-10 10:00:00|
|97500      |2023-03-12 13:20:14|200       |2023-03-12 13:00:00|
|97500      |2020-11-19 15:06:58|200       |2020-11-19 15:00:00|
|97500      |2021-07-24 17:00:33|200       |2021-07-24 17:00:00|
|98600      |2022-08-12 08:37:43|200       |2022-08-12 08:00:00|
|98600      |2023-02-05 20:20:09|200       |2023-02-05 20:00:00|
|9170       |2021-10-31 15:42:28|200       |2021-10-31 15:00:00|
|9170       |2022-10-30 12:45:35|200       |2022-10-30 12:00:00|
|9170       |2023-08-08 16:35:16|200       |2023-08-08 16:00:00|
|9171       |2023-04-12 21:56:43|200       |2023-04-12 21:00:00|
+-----------+-------------------+----------+-------------------+
only showing top 10 rows


In [10]:
weather_df.select("datetime_hour").show(10)

+-------------------+
|      datetime_hour|
+-------------------+
|2020-10-10 10:00:00|
|2023-03-12 13:00:00|
|2020-11-19 15:00:00|
|2021-07-24 17:00:00|
|2022-08-12 08:00:00|
|2023-02-05 20:00:00|
|2021-10-31 15:00:00|
|2022-10-30 12:00:00|
|2023-08-08 16:00:00|
|2023-04-12 21:00:00|
+-------------------+
only showing top 10 rows


In [ ]:

df_fact_raw = df_operational_treasure_log.alias("log") \
    .join(df_user_with_location.alias("u"), "user_id", "left") \
    .join(dim_user.alias("du").select("user_id", "user_sk"), "user_id", "left") \
    .join(df_operational_treasure.alias("t_op"), "treasure_id", "left") \
    .join(dim_treasure.alias("dt"), ["difficulty", "terrain", "stage_count"], "left") \
    .join(dim_date.withColumnRenamed("month", "dim_month"), "full_date", "left")



from pyspark.sql.functions import concat_ws, to_timestamp, col, date_trunc






In [ ]:
df_fact_raw.show()

In [60]:
# Step 1: Map weather → SK
weather_df = weather_df.withColumn(
    "weather_sk",
    when(col("weather_id").between(200, 699), 1)
    .when(col("weather_id").isNull(), 3)
    .otherwise(2)
)

df_fact_with_weather = df_fact_raw.join(
    weather_df,
    (F.to_date(df_fact_raw.log_hour) == F.to_date(weather_df.datetime_hour)) &
    (df_fact_raw.postal_code == weather_df.postal_code),
    "left"
).fillna({"weather_sk": 3})  # default "no weather"



# Step 4: Final fact
df_fact_final = df_fact_with_weather.select(
    F.col("user_sk").cast(LongType()),
    F.col("treasure_sk").alias("treasure_type_sk").cast(IntegerType()),
    F.col("datesk").alias("date_sk").cast(LongType()),
    F.when(F.col("dim_month").isin(12, 1, 2), 1)
     .when(F.col("dim_month").isin(3, 4, 5), 2)
     .when(F.col("dim_month").isin(6, 7, 8), 3)
     .when(F.col("dim_month").isin(9, 10, 11), 4)
     .cast(IntegerType()).alias("season_sk"),
    F.col("log_hour").cast(IntegerType()),
    F.col("weather_sk").cast(LongType()),
    F.col("duration_minutes").cast(IntegerType())
).fillna(0)

df_fact_final.show(10, truncate=False)


+-------+----------------+-------+---------+----------+----------+----------------+
|user_sk|treasure_type_sk|date_sk|season_sk|log_hour  |weather_sk|duration_minutes|
+-------+----------------+-------+---------+----------+----------+----------------+
|262443 |105             |601    |2        |1651708800|3         |58              |
|7924   |0               |373    |4        |1632074400|3         |24              |
|120362 |156             |161    |1        |1613743200|3         |103             |
|193280 |99              |601    |2        |1651723200|3         |88              |
|462258 |121             |874    |1        |1675360800|3         |2               |
|212883 |0               |1058   |3        |1691254800|3         |131             |
|443320 |118             |486    |1        |1641808800|3         |4               |
|216696 |105             |238    |2        |1620406800|3         |33              |
|74161  |105             |715    |3        |1661634000|3         |109       

In [ ]:
df_fact_final.groupBy("weather_sk").count().show()


In [ ]:
# Write to the table
df_fact_final.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable("factTreasureHunt")

print("Fact table loaded successfully!")

In [ ]:
#spark.stop()

In [ ]:
#%reset

Incremental Load not completly working yet

In [ ]:
before_count = spark.table("factTreasureHunt").count()
print("Rows before merge:", before_count)

In [ ]:
factTreasure_new = df_fact_final.withColumn(
    "MD5",
    F.md5(
        F.concat_ws(
            "||",
            F.col("user_sk").cast("string"),
            F.col("treasure_type_sk").cast("string"),
            F.col("date_sk").cast("string"),
            F.col("season_sk").cast("string"),
            F.col("log_hour").cast("string"),
            F.col("weather_sk").cast("string"),
            F.col("duration_minutes").cast("string")
        )
    )
)


In [ ]:
#DEBUG
from pyspark.sql import functions as F

factTreasure_new.groupBy("user_sk", "treasure_type_sk", "date_sk", "log_hour") \
    .agg(F.count("*").alias("cnt")) \
    .filter(F.col("cnt") > 1) \
    .show(100, truncate=False)

In [ ]:
from pyspark.sql.window import Window

window_spec = Window.partitionBy("user_sk", "treasure_type_sk", "date_sk", "log_hour") \
                    .orderBy(F.col("MD5").desc())

factTreasure_new_dedup = factTreasure_new.withColumn(
    "rn", F.row_number().over(window_spec)
).filter(F.col("rn") == 1).drop("rn")

In [ ]:
spark.sql("""
ALTER TABLE factTreasureHunt
ADD COLUMNS (MD5 STRING)
""")

In [ ]:
fact_current = spark.table("factTreasureHunt")
fact_current.createOrReplaceTempView("factTreasure_current")

In [ ]:
factTreasure_new_dedup.createOrReplaceTempView("factTreasure_new")

In [ ]:
spark.sql("""
MERGE INTO factTreasure_current AS target
USING factTreasure_new AS source
ON  target.user_sk = source.user_sk
AND target.treasure_type_sk = source.treasure_type_sk
AND target.date_sk = source.date_sk
AND target.log_hour = source.log_hour

WHEN MATCHED AND NOT (source.MD5 <=> target.MD5) THEN
  UPDATE SET *

WHEN NOT MATCHED THEN
  INSERT *
""")

In [ ]:
# Check counts
after_count = spark.table("factTreasureHunt").count()
print("Inserted rows:", after_count - before_count)


In [ ]:
# 1. Read the latest data from the treasure and related tables
df_operational_treasure = spark.read \
    .format("jdbc") \
    .option("url", "jdbc:postgresql://postgres-data4:5432/catchem") \
    .option("dbtable", "treasure") \
    .option("user", "postgres") \
    .option("password", "data") \
    .option("driver", "org.postgresql.Driver") \
    .load()

# 2. Join with other relevant tables to build df_fact_final
# (users, weather, logs, etc.) – same as your normal ETL logic

# 3. Recompute MD5
from pyspark.sql import functions as F

factTreasure_new = df_fact_final.withColumn(
    "MD5",
    F.md5(F.concat_ws(
        "||",
        F.col("user_sk").cast("string"),
        F.col("treasure_type_sk").cast("string"),
        F.col("date_sk").cast("string"),
        F.col("season_sk").cast("string"),
        F.col("log_hour").cast("string"),
        F.col("weather_sk").cast("string"),
        F.col("duration_minutes").cast("string")
    ))
)

from pyspark.sql.window import Window
import pyspark.sql.functions as F

window_spec = Window.partitionBy("user_sk", "treasure_type_sk", "date_sk", "log_hour") \
                    .orderBy(F.col("MD5").desc())  # or timestamp if you have one

factTreasure_new_dedup = factTreasure_new.withColumn(
    "rn", F.row_number().over(window_spec)
).filter(F.col("rn") == 1).drop("rn")

# Define composite key for comparison
key_cols = ["user_sk", "treasure_type_sk", "date_sk", "log_hour"]

# Anti-join: keep only rows not in fact_current
factTreasure_incremental = factTreasure_new_dedup.join(
    fact_current.select(*key_cols),
    on=key_cols,
    how="left_anti"   # keeps only rows NOT in fact_current
)

factTreasure_incremental.createOrReplaceTempView("factTreasure_new")

# 4. Run your merge
merge_result = spark.sql("""
MERGE INTO factTreasure_current AS target
USING factTreasure_new AS source
ON target.user_sk = source.user_sk
AND target.treasure_type_sk = source.treasure_type_sk
AND target.date_sk = source.date_sk
AND target.log_hour = source.log_hour
WHEN MATCHED AND NOT (source.MD5 <=> target.MD5) THEN
  UPDATE SET *
WHEN NOT MATCHED THEN
  INSERT *
""")

In [ ]:
after_count = spark.table("factTreasureHunt").count()
print("Rows after merge:", after_count)